# Biohub Local Scorer and Parameter Sweep

Bu notebook EDA degil; secilen validation set uzerinde baseline detection/linking skorlarini olcer ve parameter sweep yapar. Metric resmi Kaggle metric'in yaklasik local versiyonudur; sparse GT nedeniyle hem matched-edge hem naive edge skorlarini raporlar.

## 1. Runtime ve Importlar

In [ ]:
from pathlib import Path
import itertools
import json
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive already unavailable.')

try:
    import zarr  # noqa: F401
except Exception:
    print('Installing missing zarr dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'zarr', 'numcodecs'])

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

## 2. Proje, Dataset ve Validation Set

In [ ]:
PROJECT_CANDIDATES = [
    Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development'),
    Path.cwd(),
]

PROJECT_DIR = None
for p in PROJECT_CANDIDATES:
    if (p / 'biohub_baseline.py').exists():
        PROJECT_DIR = p
        break

if PROJECT_DIR is None:
    raise FileNotFoundError('biohub_baseline.py bulunamadi. Notebook proje veya Drive proje klasorunden calismali.')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import biohub_baseline as bh


def find_dataset_base():
    candidates = [
        PROJECT_DIR / 'data',
        PROJECT_DIR / 'data' / 'biohub-cell-tracking-during-development',
        Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
        Path('/kaggle/input/biohub-cell-tracking-during-development'),
    ]
    for c in candidates:
        if (c / 'train').exists() and (c / 'test').exists():
            return c
    raise FileNotFoundError('train/test iceren dataset base bulunamadi.')


BASE_DIR = find_dataset_base()
TRAIN_DIR = BASE_DIR / 'train'
TEST_DIR = BASE_DIR / 'test'
OUTPUT_DIR = PROJECT_DIR / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_path = OUTPUT_DIR / 'validation_samples.csv'
if validation_path.exists():
    validation_df = pd.read_csv(validation_path)
else:
    fallback_ids = ['44b6_0c582fdc', '6bba_afb141ff', '6bba_09961292']
    validation_df = pd.DataFrame({'sample_id': fallback_ids})
    print('validation_samples.csv bulunamadi; fallback sample list kullaniliyor.')

print('PROJECT_DIR:', PROJECT_DIR)
print('BASE_DIR:', BASE_DIR)
print('validation samples:', len(validation_df))
display(validation_df.head())

## 3. Sweep Ayarlari

`MAX_TIMEPOINTS=20` hizli GT-aligned sanity icin. Full validation icin `MAX_TIMEPOINTS=None` yap.

In [ ]:
RUN_SINGLE_SAMPLE_SANITY = True
RUN_PARAMETER_SWEEP = False

# Quick mod: GT'nin basladigi yerden 20 timepoint. Full sample icin MAX_TIMEPOINTS=None yap.
MAX_TIMEPOINTS = 20
USE_GT_ALIGNED_WINDOW = True

# Ilk sweep kucuk tutuldu. Genisletirken tek tek buyut.
SWEEP_SAMPLE_IDS = validation_df['sample_id'].head(6).tolist()

DETECTION_GRID = [
    {'percentile_high': 99.7, 'threshold_abs': 0.18, 'gaussian_sigma': 1.0, 'min_distance': 3},
    {'percentile_high': 99.8, 'threshold_abs': 0.20, 'gaussian_sigma': 1.0, 'min_distance': 3},
    {'percentile_high': 99.9, 'threshold_abs': 0.20, 'gaussian_sigma': 1.0, 'min_distance': 3},
    {'percentile_high': 99.8, 'threshold_abs': 0.24, 'gaussian_sigma': 1.0, 'min_distance': 3},
    {'percentile_high': 99.8, 'threshold_abs': 0.20, 'gaussian_sigma': 1.2, 'min_distance': 4},
]

LINKING_GRID = [5.0, 6.0, 7.0, 8.0, 9.0]

print('SWEEP_SAMPLE_IDS:', SWEEP_SAMPLE_IDS)
print('detection configs:', len(DETECTION_GRID))
print('linking radii:', LINKING_GRID)

## 4. Local Metric Helpers

In [ ]:
SCALES_ZYX = np.array([1.625, 0.40625, 0.40625], dtype=float)
MATCH_MAX_DISTANCE_UM = 7.0


def add_param_defaults(params):
    merged = dict(bh.DETECTION_PARAMS)
    merged.update(params)
    return merged


def edge_set_from_df(edges_df):
    if edges_df is None or len(edges_df) == 0:
        return set()
    return set((int(r.source_id), int(r.target_id)) for r in edges_df.itertuples(index=False))


def match_pred_to_gt_by_time(pred_df, gt_nodes_df, max_distance_um=MATCH_MAX_DISTANCE_UM):
    rows = []
    pred_to_gt = {}
    gt_to_pred = {}

    if pred_df.empty or gt_nodes_df.empty:
        return pd.DataFrame(columns=['t', 'pred_node_id', 'gt_node_id', 'distance_um']), pred_to_gt, gt_to_pred

    common_t = sorted(set(pred_df['t'].astype(int)) & set(gt_nodes_df['t'].astype(int)))
    for t in common_t:
        pred_t = pred_df[pred_df['t'].astype(int) == int(t)].reset_index(drop=True)
        gt_t = gt_nodes_df[gt_nodes_df['t'].astype(int) == int(t)].reset_index(drop=True)
        if pred_t.empty or gt_t.empty:
            continue

        pred_um = pred_t[['z', 'y', 'x']].to_numpy(dtype=float) * SCALES_ZYX
        gt_um = gt_t[['z', 'y', 'x']].to_numpy(dtype=float) * SCALES_ZYX
        dist = cdist(pred_um, gt_um)
        pred_idx, gt_idx = linear_sum_assignment(dist)

        for pi, gi in zip(pred_idx, gt_idx):
            d = float(dist[pi, gi])
            if d <= max_distance_um:
                pred_id = int(pred_t.loc[pi, 'node_id'])
                gt_id = int(gt_t.loc[gi, 'node_id'])
                pred_to_gt[pred_id] = gt_id
                gt_to_pred[gt_id] = pred_id
                rows.append({'t': int(t), 'pred_node_id': pred_id, 'gt_node_id': gt_id, 'distance_um': d})

    return pd.DataFrame(rows), pred_to_gt, gt_to_pred


def score_prediction_against_sparse_gt(pred_df, pred_edges_df, gt_nodes_df, gt_edges_df, eval_t_min=None, eval_t_max=None):
    gt_nodes_eval = gt_nodes_df.copy()
    gt_edges_eval = gt_edges_df.copy()
    pred_eval = pred_df.copy()
    pred_edges_eval = pred_edges_df.copy()

    if eval_t_min is not None or eval_t_max is not None:
        if eval_t_min is None:
            eval_t_min = 0
        if eval_t_max is None:
            eval_t_max = 10**9
        gt_nodes_eval = gt_nodes_eval[
            (gt_nodes_eval['t'].astype(int) >= int(eval_t_min)) & (gt_nodes_eval['t'].astype(int) <= int(eval_t_max))
        ].copy()
        pred_eval = pred_eval[
            (pred_eval['t'].astype(int) >= int(eval_t_min)) & (pred_eval['t'].astype(int) <= int(eval_t_max))
        ].copy()
        valid_gt = set(gt_nodes_eval['node_id'].astype(int))
        valid_pred = set(pred_eval['node_id'].astype(int))
        gt_edges_eval = gt_edges_eval[
            gt_edges_eval['source_id'].astype(int).isin(valid_gt) & gt_edges_eval['target_id'].astype(int).isin(valid_gt)
        ].copy()
        pred_edges_eval = pred_edges_eval[
            pred_edges_eval['source_id'].astype(int).isin(valid_pred) & pred_edges_eval['target_id'].astype(int).isin(valid_pred)
        ].copy()

    matches_df, pred_to_gt, gt_to_pred = match_pred_to_gt_by_time(pred_eval, gt_nodes_eval)

    gt_edges = edge_set_from_df(gt_edges_eval)
    pred_edge_rows = []
    edge_tp = 0
    matched_pred_edges = 0
    for row in pred_edges_eval.itertuples(index=False):
        src = int(row.source_id)
        tgt = int(row.target_id)
        if src in pred_to_gt and tgt in pred_to_gt:
            matched_pred_edges += 1
            gt_edge = (pred_to_gt[src], pred_to_gt[tgt])
            is_tp = gt_edge in gt_edges
            edge_tp += int(is_tp)
            pred_edge_rows.append({'source_id': src, 'target_id': tgt, 'gt_source_id': gt_edge[0], 'gt_target_id': gt_edge[1], 'is_tp': is_tp})

    edge_fn = max(0, len(gt_edges) - edge_tp)
    edge_fp_matched = max(0, matched_pred_edges - edge_tp)
    edge_fp_naive = max(0, len(pred_edges_eval) - edge_tp)

    matched_jaccard = edge_tp / max(1, edge_tp + edge_fp_matched + edge_fn)
    naive_jaccard = edge_tp / max(1, edge_tp + edge_fp_naive + edge_fn)

    return {
        'pred_nodes': int(len(pred_eval)),
        'gt_nodes_sparse': int(len(gt_nodes_eval)),
        'matched_nodes': int(len(matches_df)),
        'node_recall_sparse_gt': float(len(matches_df) / max(1, len(gt_nodes_eval))),
        'node_precision_sparse_gt': float(len(matches_df) / max(1, len(pred_eval))),
        'pred_edges': int(len(pred_edges_eval)),
        'gt_edges_sparse': int(len(gt_edges)),
        'matched_pred_edges': int(matched_pred_edges),
        'edge_tp': int(edge_tp),
        'edge_fp_matched': int(edge_fp_matched),
        'edge_fp_naive': int(edge_fp_naive),
        'edge_fn': int(edge_fn),
        'edge_jaccard_matched': float(matched_jaccard),
        'edge_jaccard_naive': float(naive_jaccard),
        'match_distance_median_um': float(matches_df['distance_um'].median()) if len(matches_df) else np.nan,
        'match_distance_p95_um': float(matches_df['distance_um'].quantile(0.95)) if len(matches_df) else np.nan,
    }


def choose_eval_window(gt_nodes_df, img_shape, max_timepoints=MAX_TIMEPOINTS, use_gt_aligned=USE_GT_ALIGNED_WINDOW):
    if max_timepoints is None:
        return 0, int(img_shape[0]) - 1
    window = int(max_timepoints)
    if use_gt_aligned and len(gt_nodes_df):
        start_t = int(gt_nodes_df['t'].min())
    else:
        start_t = 0
    end_t = min(int(img_shape[0]) - 1, start_t + window - 1)
    return start_t, end_t


def detect_cells_for_sample_window(img, sample_id, params, t_start, t_end):
    per_frame = []
    counts = []
    for t in tqdm(range(int(t_start), int(t_end) + 1), desc=f'Detecting {sample_id} [{t_start}-{t_end}]'):
        volume_t = np.asarray(img[t])
        frame_df = bh.detect_cells_in_timepoint(volume_t, t=t, params=params)
        counts.append(len(frame_df))
        per_frame.append(frame_df)
    detections = pd.concat(per_frame, ignore_index=True) if per_frame else pd.DataFrame(columns=['t', 'z', 'y', 'x', 'score'])
    detections.insert(0, 'sample_id', sample_id)
    detections.insert(1, 'node_id', np.arange(1, len(detections) + 1, dtype=int))
    if counts:
        print(
            f'{sample_id}: detections total={len(detections)}, '
            f'per-frame min/median/max={np.min(counts)}/{np.median(counts):.1f}/{np.max(counts)}'
        )
    return detections


def run_one_sample(sample_id, detection_params, linking_radius_um=7.0, max_timepoints=MAX_TIMEPOINTS):
    img = bh.open_image_zarr(TRAIN_DIR / f'{sample_id}.zarr', print_info=False)
    gt_nodes_df, gt_edges_df = bh.load_geff_annotations(TRAIN_DIR / f'{sample_id}.geff')
    eval_t_min, eval_t_max = choose_eval_window(gt_nodes_df, img.shape, max_timepoints=max_timepoints)
    print(f'{sample_id}: eval window t={eval_t_min}..{eval_t_max}')
    start = time.time()
    if max_timepoints is None and eval_t_min == 0 and eval_t_max == int(img.shape[0]) - 1:
        pred_df = bh.detect_cells_for_sample(img, sample_id, detection_params, max_timepoints=None)
    else:
        pred_df = detect_cells_for_sample_window(img, sample_id, detection_params, eval_t_min, eval_t_max)
    detect_seconds = time.time() - start

    link_params = dict(bh.LINKING_PARAMS)
    link_params['max_distance_um'] = float(linking_radius_um)
    start = time.time()
    pred_edges_df = bh.link_detections(pred_df, link_params)
    link_seconds = time.time() - start

    score = score_prediction_against_sparse_gt(pred_df, pred_edges_df, gt_nodes_df, gt_edges_df, eval_t_min=eval_t_min, eval_t_max=eval_t_max)
    score.update({
        'sample_id': sample_id,
        'eval_t_min': int(eval_t_min),
        'eval_t_max': int(eval_t_max),
        'max_timepoints': int(eval_t_max - eval_t_min + 1),
        'linking_radius_um': float(linking_radius_um),
        'detect_seconds': float(detect_seconds),
        'link_seconds': float(link_seconds),
    })
    return score

## 5. Single-Sample Sanity

In [ ]:
if RUN_SINGLE_SAMPLE_SANITY:
    sanity_sample_id = SWEEP_SAMPLE_IDS[0]
    sanity_params = add_param_defaults(DETECTION_GRID[1])
    sanity_score = run_one_sample(sanity_sample_id, sanity_params, linking_radius_um=7.0, max_timepoints=MAX_TIMEPOINTS)
    display(pd.DataFrame([sanity_score]))
else:
    print('RUN_SINGLE_SAMPLE_SANITY=False')

## 6. Parameter Sweep

In [ ]:
def run_sweep(sample_ids, detection_grid, linking_grid, max_timepoints=MAX_TIMEPOINTS):
    rows = []
    total = len(sample_ids) * len(detection_grid) * len(linking_grid)
    pbar = tqdm(total=total, desc='local sweep')
    for det_idx, det_overrides in enumerate(detection_grid):
        det_params = add_param_defaults(det_overrides)
        for radius in linking_grid:
            for sample_id in sample_ids:
                try:
                    score = run_one_sample(sample_id, det_params, linking_radius_um=radius, max_timepoints=max_timepoints)
                    score.update({
                        'det_config_id': det_idx,
                        'percentile_high': det_params['percentile_high'],
                        'threshold_abs': det_params['threshold_abs'],
                        'gaussian_sigma': det_params['gaussian_sigma'],
                        'min_distance': det_params['min_distance'],
                    })
                    rows.append(score)
                except Exception as exc:
                    rows.append({'sample_id': sample_id, 'det_config_id': det_idx, 'linking_radius_um': radius, 'error': repr(exc)})
                pbar.update(1)
    pbar.close()
    return pd.DataFrame(rows)


if RUN_PARAMETER_SWEEP:
    sweep_df = run_sweep(SWEEP_SAMPLE_IDS, DETECTION_GRID, LINKING_GRID, max_timepoints=MAX_TIMEPOINTS)
    sweep_path = OUTPUT_DIR / 'local_sweep_results.csv'
    sweep_df.to_csv(sweep_path, index=False)
    print('saved:', sweep_path)
    display(sweep_df.head())
else:
    print('RUN_PARAMETER_SWEEP=False. Acmak icin flagi True yap.')

## 7. Sweep Sonuc Ozeti

In [ ]:
sweep_path = OUTPUT_DIR / 'local_sweep_results.csv'
if sweep_path.exists():
    sweep_df = pd.read_csv(sweep_path)
    ok_df = sweep_df[sweep_df.get('error', pd.Series(index=sweep_df.index, dtype=object)).isna()].copy()
    metric_cols = ['edge_jaccard_matched', 'edge_jaccard_naive', 'node_recall_sparse_gt', 'matched_nodes', 'pred_nodes', 'pred_edges', 'detect_seconds', 'link_seconds']
    summary_df = ok_df.groupby(['det_config_id', 'percentile_high', 'threshold_abs', 'gaussian_sigma', 'min_distance', 'linking_radius_um'])[metric_cols].mean().reset_index()
    summary_df = summary_df.sort_values(['edge_jaccard_matched', 'node_recall_sparse_gt'], ascending=False)
    display(summary_df.head(20))

    plt.figure(figsize=(9, 4))
    for det_id, sub in summary_df.groupby('det_config_id'):
        sub = sub.sort_values('linking_radius_um')
        plt.plot(sub['linking_radius_um'], sub['edge_jaccard_matched'], marker='o', label=f'det {det_id}')
    plt.xlabel('linking_radius_um')
    plt.ylabel('edge_jaccard_matched')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print('No sweep file yet:', sweep_path)

## 8. Selected Config Validation

First sweep suggests config `0` is highest recall/edge, while config `4` is more conservative. Run these on the full 24-sample validation set before choosing a submission config.

In [ ]:
RUN_SELECTED_CONFIG_VALIDATION = False

SELECTED_SAMPLE_IDS = validation_df['sample_id'].tolist()
SELECTED_CONFIGS = [
    {
        'selected_config_name': 'det0_radius8_high_recall',
        'detection': {'percentile_high': 99.7, 'threshold_abs': 0.18, 'gaussian_sigma': 1.0, 'min_distance': 3},
        'linking_radius_um': 8.0,
    },
    {
        'selected_config_name': 'det0_radius9_high_recall',
        'detection': {'percentile_high': 99.7, 'threshold_abs': 0.18, 'gaussian_sigma': 1.0, 'min_distance': 3},
        'linking_radius_um': 9.0,
    },
    {
        'selected_config_name': 'det4_radius8_balanced',
        'detection': {'percentile_high': 99.8, 'threshold_abs': 0.20, 'gaussian_sigma': 1.2, 'min_distance': 4},
        'linking_radius_um': 8.0,
    },
    {
        'selected_config_name': 'det4_radius9_balanced',
        'detection': {'percentile_high': 99.8, 'threshold_abs': 0.20, 'gaussian_sigma': 1.2, 'min_distance': 4},
        'linking_radius_um': 9.0,
    },
]


def run_selected_config_validation(sample_ids=SELECTED_SAMPLE_IDS, selected_configs=SELECTED_CONFIGS, max_timepoints=MAX_TIMEPOINTS):
    rows = []
    total = len(sample_ids) * len(selected_configs)
    pbar = tqdm(total=total, desc='selected config validation')
    for cfg in selected_configs:
        det_params = add_param_defaults(cfg['detection'])
        for sample_id in sample_ids:
            try:
                score = run_one_sample(sample_id, det_params, linking_radius_um=cfg['linking_radius_um'], max_timepoints=max_timepoints)
                score.update({
                    'selected_config_name': cfg['selected_config_name'],
                    'percentile_high': det_params['percentile_high'],
                    'threshold_abs': det_params['threshold_abs'],
                    'gaussian_sigma': det_params['gaussian_sigma'],
                    'min_distance': det_params['min_distance'],
                })
                if 'estimated_number_of_nodes' in validation_df.columns:
                    meta = validation_df.set_index('sample_id').loc[sample_id]
                    estimated = float(meta['estimated_number_of_nodes'])
                    expected_window_nodes = estimated * score['max_timepoints'] / 100.0
                    score['estimated_number_of_nodes'] = estimated
                    score['expected_nodes_window'] = expected_window_nodes
                    if score['pred_nodes'] > 0:
                        score['node_overprediction_factor'] = min(1.0, expected_window_nodes / score['pred_nodes'])
                    else:
                        score['node_overprediction_factor'] = 0.0
                    score['edge_jaccard_adjusted_approx'] = score['edge_jaccard_matched'] * score['node_overprediction_factor']
                rows.append(score)
            except Exception as exc:
                rows.append({'sample_id': sample_id, 'selected_config_name': cfg['selected_config_name'], 'error': repr(exc)})
            pbar.update(1)
    pbar.close()
    return pd.DataFrame(rows)


if RUN_SELECTED_CONFIG_VALIDATION:
    selected_df = run_selected_config_validation()
    selected_path = OUTPUT_DIR / 'selected_config_validation.csv'
    selected_df.to_csv(selected_path, index=False)
    print('saved:', selected_path)
    display(selected_df.head())
else:
    print('RUN_SELECTED_CONFIG_VALIDATION=False. Acmak icin flagi True yap.')

## 9. Selected Config Summary

In [ ]:
selected_path = OUTPUT_DIR / 'selected_config_validation.csv'
if selected_path.exists():
    selected_df = pd.read_csv(selected_path)
    ok_df = selected_df[selected_df.get('error', pd.Series(index=selected_df.index, dtype=object)).isna()].copy()
    if 'edge_jaccard_adjusted_approx' not in ok_df.columns and 'estimated_number_of_nodes' in validation_df.columns:
        meta_df = validation_df[['sample_id', 'estimated_number_of_nodes']].copy()
        ok_df = ok_df.merge(meta_df, on='sample_id', how='left')
        ok_df['expected_nodes_window'] = ok_df['estimated_number_of_nodes'] * ok_df['max_timepoints'] / 100.0
        ok_df['node_overprediction_factor'] = np.minimum(1.0, ok_df['expected_nodes_window'] / ok_df['pred_nodes'].clip(lower=1))
        ok_df['edge_jaccard_adjusted_approx'] = ok_df['edge_jaccard_matched'] * ok_df['node_overprediction_factor']
    metric_cols = ['edge_jaccard_matched', 'edge_jaccard_adjusted_approx', 'node_overprediction_factor', 'edge_jaccard_naive', 'node_recall_sparse_gt', 'matched_nodes', 'pred_nodes', 'pred_edges', 'detect_seconds', 'link_seconds']
    metric_cols = [c for c in metric_cols if c in ok_df.columns]
    selected_summary_df = ok_df.groupby(['selected_config_name', 'percentile_high', 'threshold_abs', 'gaussian_sigma', 'min_distance', 'linking_radius_um'])[metric_cols].mean().reset_index()
    sort_cols = ['edge_jaccard_adjusted_approx', 'edge_jaccard_matched'] if 'edge_jaccard_adjusted_approx' in selected_summary_df.columns else ['edge_jaccard_matched', 'node_recall_sparse_gt']
    selected_summary_df = selected_summary_df.sort_values(sort_cols, ascending=False)
    display(selected_summary_df)
else:
    print('No selected config validation file yet:', selected_path)

## 10. Next Run Plan

1. Run `RUN_SELECTED_CONFIG_VALIDATION=True` on the 24-sample validation set with `MAX_TIMEPOINTS=20`.
2. Add approximate node overprediction adjustment.
3. For final validation, set `MAX_TIMEPOINTS=None` on the best configs only.
4. Add division candidates after the edge baseline is stable.
5. Use results to choose the first real submission notebook config.